# Notebook 18 — Sepsis Progression Model Training redo





## Load train / validation / test datasets

In [1]:
import pandas as pd
import numpy as np

train_dataset = pd.read_csv("../data/processed/sepsis_progression_train.csv")
validation_dataset = pd.read_csv("../data/processed/sepsis_progression_validation.csv")
test_dataset = pd.read_csv("../data/processed/sepsis_progression_test.csv")

print("Train shape:", train_dataset.shape)
print("Validation shape:", validation_dataset.shape)
print("Test shape:", test_dataset.shape)

Train shape: (300569, 122)
Validation shape: (64153, 122)
Test shape: (64226, 122)


In [2]:
print("Columns:", train_dataset.columns.tolist())

sofa_cols = [c for c in train_dataset.columns if "sofa" in c.lower()]
print("\nSOFA-related columns found:", sofa_cols)
assert len(sofa_cols) > 0, "SOFA features missing — rerun fixed Notebook 17 before continuing"

lab_ffill_cols = [c for c in train_dataset.columns if c.endswith("_ffill")]
print("Lab forward-fill columns found:", len(lab_ffill_cols))
assert len(lab_ffill_cols) > 0, "Lab ffill features missing — rerun fixed Notebook 17 before continuing"

print("\nTarget distribution:")
print(train_dataset["progression_class"].value_counts().sort_index())

Columns: ['hour', 'heart_rate', 'resp_rate', 'temperature', 'sbp', 'dbp', 'mbp', 'spo2', 'gcs', 'creatinine', 'bun', 'urineoutput_last', 'urineoutput_sum', 'urineoutput_24hr', 'wbc', 'hemoglobin', 'hematocrit', 'platelet', 'bands', 'sodium', 'potassium', 'chloride', 'bicarbonate', 'calcium', 'magnesium', 'aniongap', 'albumin', 'bilirubin_total', 'bilirubin_max', 'inr', 'pt', 'ptt', 'crp', 'lactate', 'pao2fio2ratio_novent', 'pao2fio2ratio_vent', 'glucose_lab', 'shock_index', 'map_calculated', 'bun_creatinine_ratio', 'spo2_deficit', 'heart_rate_prev', 'heart_rate_delta', 'resp_rate_prev', 'resp_rate_delta', 'temperature_prev', 'temperature_delta', 'sbp_prev', 'sbp_delta', 'mbp_prev', 'mbp_delta', 'spo2_prev', 'spo2_delta', 'gcs_prev', 'gcs_delta', 'heart_rate_roll3_mean', 'heart_rate_roll6_mean', 'sofa_score', 'sofa_prev', 'sofa_delta_1h', 'sofa_roll3_mean', 'sofa_roll3_max', 'sofa_roll3_min', 'sofa_roll6_mean', 'sofa_roll6_max', 'respiration', 'coagulation', 'liver', 'cardiovascular', '

In [3]:
missing_summary = pd.DataFrame({
    "missing_count": train_dataset.isna().sum(),
    "missing_percent": train_dataset.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

print("Worst 20 remaining missing columns (raw lab columns expected here, ffill versions should be far lower):")
print(missing_summary.head(20))

Worst 20 remaining missing columns (raw lab columns expected here, ffill versions should be far lower):
                      missing_count  missing_percent
crp                          300326        99.919153
bands                        298750        99.394815
pao2fio2ratio_novent         297596        99.010876
crp_hours_since              296422        98.620284
crp_ffill                    296422        98.620284
albumin                      295957        98.465577
bilirubin_max                291407        96.951781
bilirubin_total              291401        96.949785
liver                        291365        96.937808
gcs_delta                    283778        94.413596
inr                          282679        94.047956
pt                           282674        94.046292
ptt                          282256        93.907223
pao2fio2ratio_vent           278294        92.589056
lactate                      278106        92.526508
calcium                      276337        91.93

In [4]:
non_numeric = train_dataset.select_dtypes(exclude=["number"]).columns.tolist()
print("Non-numeric columns:", non_numeric)

Non-numeric columns: []


##  Define features / target



In [5]:
target_column = "progression_class"

raw_lab_cols_replaced = [
    "creatinine", "bun", "wbc", "hemoglobin", "hematocrit", "platelet", "bands",
    "sodium", "potassium", "chloride", "bicarbonate", "calcium", "magnesium",
    "aniongap", "albumin", "bilirubin_total", "bilirubin_max", "inr", "pt", "ptt",
    "crp", "lactate", "glucose_lab", "pao2fio2ratio_novent", "pao2fio2ratio_vent"
]
raw_lab_cols_replaced = [c for c in raw_lab_cols_replaced if c in train_dataset.columns]

feature_columns = [
    c for c in train_dataset.columns
    if c != target_column and c not in raw_lab_cols_replaced
]

X_train = train_dataset[feature_columns].copy()
y_train = train_dataset[target_column].copy()

X_val = validation_dataset[feature_columns].copy()
y_val = validation_dataset[target_column].copy()

X_test = test_dataset[feature_columns].copy()
y_test = test_dataset[target_column].copy()

print("Number of features:", len(feature_columns))
print("X_train:", X_train.shape, " X_val:", X_val.shape, " X_test:", X_test.shape)

Number of features: 96
X_train: (300569, 96)  X_val: (64153, 96)  X_test: (64226, 96)


## LightGBM setup

In [6]:
import lightgbm as lgb
print("LightGBM version:", lgb.__version__)

LightGBM version: 4.6.0


##  Baseline model — unweighted


In [15]:
baseline_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

baseline_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

print("Baseline trained. Best iteration:", baseline_model.best_iteration_)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.039421 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10359
[LightGBM] [Info] Number of data points in the train set: 300569, number of used features: 96
[LightGBM] [Info] Start training from score -1.262567
[LightGBM] [Info] Start training from score -0.718768
[LightGBM] [Info] Start training from score -1.470890
Baseline trained. Best iteration: 459


## Class-weighted model

In [8]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
weight_map = {cls: w for cls, w in zip(classes, class_weights)}
print("Balanced class weights:", weight_map)

weighted_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=31,
    class_weight=weight_map,
    random_state=42,
    n_jobs=-1
)

weighted_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

print("Weighted model trained. Best iteration:", weighted_model.best_iteration_)

Balanced class weights: {np.int64(0): np.float64(1.178161392615937), np.int64(1): np.float64(0.6839678779562589), np.int64(2): np.float64(1.4510357679068846)}
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.040108 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10359
[LightGBM] [Info] Number of data points in the train set: 300569, number of used features: 96
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
Weighted model trained. Best iteration: 1234


## Moderate-weight model

In [9]:
moderate_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=31,
    class_weight={0: 1.0, 1: 1.0, 2: 1.5},
    random_state=42,
    n_jobs=-1
)

moderate_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

print("Moderate model trained. Best iteration:", moderate_model.best_iteration_)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033108 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10359
[LightGBM] [Info] Number of data points in the train set: 300569, number of used features: 96
[LightGBM] [Info] Start training from score -1.371297
[LightGBM] [Info] Start training from score -0.827497
[LightGBM] [Info] Start training from score -1.174154
Moderate model trained. Best iteration: 684


## Compare all three on validation — accuracy, macro F1, class-2 recall/precision/F1

In [10]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

models = {
    "baseline": baseline_model,
    "weighted": weighted_model,
    "moderate": moderate_model,
}

results = []
reports = {}

for name, m in models.items():
    y_pred = m.predict(X_val)
    reports[name] = classification_report(y_val, y_pred, output_dict=True, digits=4)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_val, y_pred),
        "macro_f1": f1_score(y_val, y_pred, average="macro"),
        "class2_precision": reports[name]["2"]["precision"],
        "class2_recall": reports[name]["2"]["recall"],
        "class2_f1": reports[name]["2"]["f1-score"],
    })

results_df = pd.DataFrame(results).sort_values("class2_f1", ascending=False)
print(results_df)

      model  accuracy  macro_f1  class2_precision  class2_recall  class2_f1
1  weighted  0.591804  0.585226          0.432361       0.525261   0.474304
2  moderate  0.629557  0.600240          0.482466       0.386882   0.429419
0  baseline  0.640219  0.566037          0.558540       0.198655   0.293073


## Full classification report + confusion matrix for best model (by class-2 F1)

In [11]:
best_name = results_df.iloc[0]["model"]
best_model = models[best_name]
print("Best model on validation (class-2 F1):", best_name)

y_val_pred_best = best_model.predict(X_val)

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_best, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_best))

print("\nPredicted distribution:")
print(pd.Series(y_val_pred_best).value_counts().sort_index())
print("\nActual distribution:")
print(y_val.value_counts().sort_index())

Best model on validation (class-2 F1): weighted

Classification Report:
              precision    recall  f1-score   support

           0     0.6201    0.8203    0.7063     17927
           1     0.6905    0.4927    0.5751     31361
           2     0.4324    0.5253    0.4743     14865

    accuracy                         0.5918     64153
   macro avg     0.5810    0.6128    0.5852     64153
weighted avg     0.6110    0.5918    0.5884     64153


Confusion Matrix:
[[14705  2353   869]
 [ 6526 15453  9382]
 [ 2482  4575  7808]]

Predicted distribution:
0    23713
1    22381
2    18059
Name: count, dtype: int64

Actual distribution:
progression_class
0    17927
1    31361
2    14865
Name: count, dtype: int64


## Feature importance — confirm SOFA features are actually being used

In [12]:
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": best_model.feature_importances_
}).sort_values("importance", ascending=False)

print("Top 25 features by importance:")
print(importance_df.head(25))

sofa_rank = importance_df.reset_index(drop=True)
sofa_rank["rank"] = sofa_rank.index + 1
print("\nWhere SOFA features landed:")
print(sofa_rank[sofa_rank["feature"].str.contains("sofa", case=False)])

Top 25 features by importance:
                     feature  importance
84                 ptt_ffill        3282
56            platelet_ffill        3079
50                 wbc_ffill        2955
90         glucose_lab_ffill        2860
11          urineoutput_24hr        2565
88             lactate_ffill        2559
94  pao2fio2ratio_vent_ffill        2522
48                 bun_ffill        2387
54          hematocrit_ffill        2371
68             calcium_ffill        2323
82                  pt_ffill        2294
32                sofa_score        2266
12               shock_index        2137
46          creatinine_ffill        2109
24                  mbp_prev        2023
52          hemoglobin_ffill        2021
38           sofa_roll6_mean        1985
62           potassium_ffill        1969
22                  sbp_prev        1967
4                        sbp        1921
76     bilirubin_total_ffill        1762
23                 sbp_delta        1749
64            chloride_ffi

##  Only now — evaluate the true holdout test set (best model, once)

In [13]:
y_test_pred = best_model.predict(X_test)

print("Model used for test evaluation:", best_name)

print("\nTest Classification Report:")
print(classification_report(y_test, y_test_pred, digits=4))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nTest accuracy:", accuracy_score(y_test, y_test_pred))
print("Test macro F1:", f1_score(y_test, y_test_pred, average="macro"))

Model used for test evaluation: weighted

Test Classification Report:
              precision    recall  f1-score   support

           0     0.6212    0.8212    0.7073     18253
           1     0.6857    0.4836    0.5672     30954
           2     0.4375    0.5321    0.4802     15019

    accuracy                         0.5909     64226
   macro avg     0.5815    0.6123    0.5849     64226
weighted avg     0.6093    0.5909    0.5867     64226


Test Confusion Matrix:
[[14989  2394   870]
 [ 6581 14970  9403]
 [ 2559  4469  7991]]

Test accuracy: 0.5908821972409928
Test macro F1: 0.5848995882505322


## Save the best model

In [14]:
import joblib

model_path = "../models/sepsis_progression_lightgbm_best.pkl"
joblib.dump(best_model, model_path)
print("Saved:", model_path, "-> model used:", best_name)

Saved: ../models/sepsis_progression_lightgbm_best.pkl -> model used: weighted
